# Final Enhancement Pipeline Comparison

## Load Libraries and Paths

In [1]:
import cv2, os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

INPUT_DIR = "../data/frames/selected"
REPORT_DIR = "../results/report_evidence/pipeline_compare"

os.makedirs(REPORT_DIR, exist_ok=True)


## Load Selected Frames

In [2]:
images=[]
names=[]

MAX_FRAMES = 20

for root,dirs,files in os.walk(INPUT_DIR):

    for f in files:

        if len(images) >= MAX_FRAMES:
            break

        if f.lower().endswith((".jpg",".png",".jpeg")):

            p=os.path.join(root,f)

            img=cv2.imread(p,0)

            if img is not None:

                images.append(img)
                names.append(f)

print("Loaded frames:",len(images))

Loaded frames: 20


## Enhancement Functions (same as previous stages)

In [3]:
def gaussian_filter(img,k):
    return cv2.GaussianBlur(img,(k,k),0)

def clahe(img,clip):
    c=cv2.createCLAHE(clipLimit=clip,tileGridSize=(8,8))
    return c.apply(img)

def unsharp(img,k,a):
    blur=cv2.GaussianBlur(img,(k,k),0)
    return cv2.addWeighted(img,1+a,blur,-a,0)


## Pipeline 1 — General Pipeline

In [4]:
def pipeline_general(img):

    img = gaussian_filter(img,3)
    img = clahe(img,2.0)
    img = unsharp(img,7,1.0)

    return img


## Pipeline 2 — Conditional Pipeline

In [5]:
def pipeline_conditional(img):

    mean = np.mean(img)
    lap = cv2.Laplacian(img,cv2.CV_64F).var()

    if lap < 300:
        img = gaussian_filter(img,5)
    else:
        img = gaussian_filter(img,3)

    if mean < 90:
        img = clahe(img,3.0)
    else:
        img = clahe(img,2.0)

    img = unsharp(img,7,1.0)

    return img


## Pipeline 3 — Multi‑Scale Pipeline

In [6]:
def pipeline_multiscale(img):

    img = gaussian_filter(img,3)
    img = clahe(img,2.0)

    s1 = unsharp(img,3,0.7)
    s2 = unsharp(img,7,1.0)

    img = cv2.addWeighted(s1,0.5,s2,0.5,0)

    return img


## Run Pipelines and Collect Metrics

In [7]:
rows=[]

for img,nm in zip(images,names):

    g = pipeline_general(img)
    c = pipeline_conditional(img)
    m = pipeline_multiscale(img)

    lap_g = cv2.Laplacian(g,cv2.CV_64F).var()
    lap_c = cv2.Laplacian(c,cv2.CV_64F).var()
    lap_m = cv2.Laplacian(m,cv2.CV_64F).var()

    rows.append({
        "frame":nm,
        "general":lap_g,
        "conditional":lap_c,
        "multiscale":lap_m
    })

    # Save comparison image
    plt.figure(figsize=(12,6))

    plt.subplot(2,2,1)
    plt.imshow(img,cmap="gray")
    plt.title("Original")
    plt.axis("off")

    plt.subplot(2,2,2)
    plt.imshow(g,cmap="gray")
    plt.title("General")
    plt.axis("off")

    plt.subplot(2,2,3)
    plt.imshow(c,cmap="gray")
    plt.title("Conditional")
    plt.axis("off")

    plt.subplot(2,2,4)
    plt.imshow(m,cmap="gray")
    plt.title("Multi‑Scale")
    plt.axis("off")

    plt.tight_layout()
    plt.savefig(os.path.join(REPORT_DIR,nm+"_pipeline_compare.png"),dpi=200)
    plt.close()


## Create Comparison Table

In [8]:
df = pd.DataFrame(rows)

display(df)

csv_path = os.path.join(REPORT_DIR,"pipeline_comparison.csv")
df.to_csv(csv_path,index=False)

print("Saved CSV:",csv_path)


,frame,general,conditional,multiscale
0,alligator_cracking_01_frame_0.jpg,3699.797679,3699.797679,2738.871252
1,alligator_cracking_01_frame_110.jpg,897.267037,415.671772,671.180269
2,alligator_cracking_01_frame_1155.jpg,2785.509327,2785.509327,2057.063238
3,alligator_cracking_01_frame_175.jpg,1973.357296,1973.357296,1462.057082
4,alligator_cracking_01_frame_225.jpg,2049.902971,2049.902971,1522.374436
5,alligator_cracking_01_frame_310.jpg,3187.418393,3187.418393,2353.603806
6,alligator_cracking_01_frame_50.jpg,1006.688035,1006.688035,751.915771
7,alligator_cracking_01_frame_515.jpg,2640.869692,2640.869692,1950.132495
8,alligator_cracking_01_frame_535.jpg,3189.367044,3189.367044,2354.039527
9,alligator_cracking_01_frame_95.jpg,1035.232525,1035.232525,773.678528


Saved CSV: ../results/report_evidence/pipeline_compare\pipeline_comparison.csv


## Select Best Pipeline

In [9]:
avg = df.mean(numeric_only=True)

print("Average Scores")
print(avg)

best_pipeline = avg.idxmax()

print("Best Pipeline:",best_pipeline)


Average Scores
general        2083.602231
conditional    2059.522468
multiscale     1542.206290
dtype: float64
Best Pipeline: general
